In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go

snp = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "CSPX ETF Stock Price History.csv")
china = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "CNYA ETF Stock Price History.csv")
em = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "EIMI ETF Stock Price History.csv")
gold = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "XAD5 ETF Stock Price History.csv")
india = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "INR ETF Stock Price History.csv")
mscieurope = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "XMEU ETF Stock Price History.csv")
smallcapeurope = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "SXRJ ETF Stock Price History.csv")
ussmallcap = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "CUSS ETF Stock Price History.csv")
silver = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "investing_dot_com_transformed" / "SSLN ETF Stock Price History.csv")

dfs = [snp, china, em, gold, india, mscieurope, smallcapeurope, ussmallcap, silver]

N = len(dfs)
T = len(dfs[0]) - 1

dfs = [df[["Date", "Price", "Vol."]] for df in dfs]

# Convert Date to datetime
dfs_new = []
for df in dfs:
    df["Date"] = pd.to_datetime(df["Date"])
    dfs_new.append(df)

dfs = dfs_new

names = ["snp", "china", "em", "gold", "india", "mscieurope",
         "smallcapeurope", "ussmallcap", "silver"]

dfs_renamed = []

for name, df in zip(names, dfs):
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"])
    
    df = df.rename(columns={
        "Price": f"Price_{name}",
        "Vol.": f"Vol_{name}"
    })
    
    dfs_renamed.append(df)

dfs = dfs_renamed

# Merge all on Date
df_merged = dfs[0]
for df in dfs[1:]:
    df_merged = df_merged.merge(df, on="Date", how="inner", suffixes=("", "_x"))

# Sort by date
df_merged = df_merged.sort_values("Date")

def correct_volume(vol_value):
    if isinstance(vol_value, (float, int)):
        return vol_value
    if vol_value[-1] == "K":
        return 1000 * float(vol_value[:-1])
    if vol_value[-1] == "M":
        return 1_000_000 * float(vol_value[:-1])
    return float(vol_value)

df_merged = df_merged.set_index("Date")

price_cols = [col for col in df_merged.columns if col.startswith("Price_")]
vol_cols = [col for col in df_merged.columns if col.startswith("Vol_")]

prices_df = df_merged[price_cols].apply(pd.to_numeric, errors="coerce")
volumes_df = df_merged[vol_cols].copy()

for col in volumes_df.columns:
    volumes_df[col] = volumes_df[col].apply(correct_volume)

volumes_df = volumes_df.fillna(0.0)

prices = prices_df.values  # shape: (T+1, N)
volumes = volumes_df.values  # shape: (T+1, N)

returns = prices[1:] / prices[:-1] - 1
prices = prices[1:]
volumes = volumes[1:]         # (T, N)
dates = df_merged.index[1:]       # (T,)
print(returns.shape, prices.shape, volumes.shape, dates.shape)

(2413, 9) (2413, 9) (2413, 9) (2413,)


In [3]:
from rollingwindow_and_eval import rolling_window
from helpers import stress_values_to_dict, costs_to_dict, cash_allocs_to_dict

alpha = 0.05   # tail probability for CVaR
max_cash_alloc = 0.3
min_weight = 0.1
stepsize = 50
etf_spreads = np.array([0.0075, 0.050, 0.100, 0.035, 0.130, 0.030, 0.030, 0.050, 0.08]) / 100
risk_free = 0.03
weight_diff_rebalance = 0.10 # minimum weight difference for a portfolio rebalance
window_size = 252
validation_size = 252

lambdas = [0.1, 0.3, 0.5, 0.8, 1.1]
gammas = [0.01, 0.025, 0.04, 0.055, 0.07]
stress_corr_weights = [0.1, 0.2, 0.3]
cash_alloc_params = [0.2, 0.5, 0.8]
m_params = [
    np.array([0.33, 0.33, 0.33]),
    np.array([0.50, 0.25, 0.25]),
    np.array([0.25, 0.50, 0.25]),
    np.array([0.25, 0.25, 0.50]),
]

"""
track:
var_alpha, cvar, sharpe, sortino, mean_return, md
costs, stress values, cash allocs
"""

total_stats_momentum = np.full((9, len(gammas), len(stress_corr_weights), len(m_params)), np.nan) # gamma, stress_corr, mparams
total_stats_momentum_cvar = np.full((9, len(lambdas), len(gammas), len(m_params)), np.nan) # lambda, gamma, mparams
total_stats_er = np.full((9, len(gammas), len(stress_corr_weights), len(cash_alloc_params)), np.nan) # gamma, stress_corr, cash_alloc_param
total_stats_er_cvar = np.full((9, len(lambdas), len(gammas), len(stress_corr_weights)), np.nan) # lambda, gamma, stress_corr
total_stats_sharpe = np.full((9, len(gammas), len(stress_corr_weights), len(cash_alloc_params)), np.nan) # gamma, stress_corr, cash_alloc_param
total_stats_sharpe_cvar = np.full((9, len(lambdas), len(gammas), len(stress_corr_weights)), np.nan) # lambda, gamma, stress_corr
labels = ["ER", "ER_cvar", "sharpe", "sharpe_cvar", "momentum_based", "momentum_cvar",
    "risk_parity", "hrp_weights", "equal_weights"]

for l_idx, l in enumerate(lambdas):
    for g_idx, g in enumerate(gammas):
        for mp_idx, mp in enumerate(m_params):
            for cash_idx, cash in enumerate(cash_alloc_params):
                for stress_corr_idx, stress_corr in enumerate(stress_corr_weights):

                    df, total_costs, total_stress_values, track_pct_rebalances, track_cash_allocs, allweights = rolling_window(
                        returns, volumes, prices,
                        window_size, stepsize, validation_size,
                        g, alpha, l,
                        mp, risk_free, etf_spreads, weight_diff_rebalance,
                        stress_corr, max_cash_alloc, cash, min_weight
                        )
                    
                    mean_weights = np.mean(allweights, axis=0) # shape (num_strategies, num_assets)

                    for stratlabel in labels:
                        assert len(df.loc[stratlabel, :].values.tolist()) == 6

                    values = df.loc["ER", :].values.tolist()

                    stresses = stress_values_to_dict(labels, total_stress_values)
                    costs = costs_to_dict(labels, total_costs)
                    cash_allocs = cash_allocs_to_dict(labels, track_cash_allocs)
                    
                    total_stats_momentum[:, g_idx, stress_corr_idx, mp_idx] = (
                        df.loc["momentum_based", :].values.tolist()
                        + [stresses["momentum_based"], costs["momentum_based"], cash_allocs["momentum_based"]]
                    )

                    total_stats_momentum_cvar[:, l_idx, g_idx, mp_idx] = (
                        df.loc["momentum_cvar", :].values.tolist()
                        + [stresses["momentum_cvar"], costs["momentum_cvar"], cash_allocs["momentum_cvar"]]
                    )

                    total_stats_er[:, g_idx, stress_corr_idx, cash_idx] = (
                        df.loc["ER", :].values.tolist()
                        + [stresses["ER"], costs["ER"], cash_allocs["ER"]]
                    )

                    total_stats_er_cvar[:, l_idx, g_idx, stress_corr_idx] = (
                        df.loc["ER_cvar", :].values.tolist()
                        + [stresses["ER_cvar"], costs["ER_cvar"], cash_allocs["ER_cvar"]]
                    )

                    total_stats_sharpe[:, g_idx, stress_corr_idx, cash_idx] = (
                        df.loc["sharpe", :].values.tolist()
                        + [stresses["sharpe"], costs["sharpe"], cash_allocs["sharpe"]]
                    )

                    total_stats_sharpe_cvar[:, l_idx, g_idx, stress_corr_idx] = (
                        df.loc["sharpe_cvar", :].values.tolist()
                        + [stresses["sharpe_cvar"], costs["sharpe_cvar"], cash_allocs["sharpe_cvar"]]
                    )



C:\Users\tobia\AppData\Roaming\Python\Python311\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\tobia\AppData\Roaming\Python\Python311\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\tobia\AppData\Roaming\Python\Python311\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/resource_handle.proto. Please 

In [5]:
stats_labels = ["var_alpha", "cvar", "sharpe", "sortino", "mean_return", "md",
                "stresses", "costs", "cashallocs"]

print('STATISTICS FOR STRATEGY: EXPECTED RETURN')
STRATEGY = total_stats_er
print('x axis: gamma')
print('y axis: stress corr parameters')
for stat_idx, stat in enumerate(stats_labels):
    print('showing statistic:', stat)
    for k in range(STRATEGY.shape[-1]):
        print('results for cash allocations:', cash_alloc_params[k])
        print(STRATEGY[stat_idx, :, :, k])
        print()

print('STATISTICS FOR STRATEGY: EXPECTED RETURN + CVAR')
STRATEGY = total_stats_er_cvar
print('x axis: lambda')
print('y axis: gamma')
for stat_idx, stat in enumerate(stats_labels):
    print('showing statistic:', stat)
    for k in range(STRATEGY.shape[-1]):
        print('results for stress values:', stress_corr_weights[k])
        print(STRATEGY[stat_idx, :, :, k])
        print()

print('STATISTICS FOR STRATEGY: SHARPE')
STRATEGY = total_stats_sharpe
print('x axis: gamma')
print('y axis: stress corr parameters')
for stat_idx, stat in enumerate(stats_labels):
    print('showing statistic:', stat)
    for k in range(STRATEGY.shape[-1]):
        print('results for cash allocations:', cash_alloc_params[k])
        print(STRATEGY[stat_idx, :, :, k])
        print()

print('STATISTICS FOR STRATEGY: SHARPE + CVAR')
STRATEGY = total_stats_sharpe_cvar
print('x axis: lambda')
print('y axis: gamma')
for stat_idx, stat in enumerate(stats_labels):
    print('showing statistic:', stat)
    for k in range(STRATEGY.shape[-1]):
        print('results for stress values:', stress_corr_weights[k])
        print(STRATEGY[stat_idx, :, :, k])
        print()

print('STATISTICS FOR STRATEGY: MOMENTUM')
STRATEGY = total_stats_momentum
print('x axis: gamma')
print('y axis: stress corr parameters')
for stat_idx, stat in enumerate(stats_labels):
    print('showing statistic:', stat)
    for k in range(STRATEGY.shape[-1]):
        print('results for m-parameters:', m_params[k])
        print(STRATEGY[stat_idx, :, :, k])
        print()

print('STATISTICS FOR STRATEGY: MOMENTUM + CVAR')
STRATEGY = total_stats_sharpe_cvar
print('x axis: lambda')
print('y axis: gamma')
for stat_idx, stat in enumerate(stats_labels):
    print('showing statistic:', stat)
    for k in range(STRATEGY.shape[-1]):
        print('results for m-parameters:', m_params[k])
        print(STRATEGY[stat_idx, :, :, k])
        print()

STATISTICS FOR STRATEGY: EXPECTED RETURN
x axis: gamma
y axis: stress corr parameters
showing statistic: var_alpha
results for cash allocations: 0.2
[[-0.00840862 -0.00840827 -0.00840741]
 [-0.00834926 -0.00834811 -0.00834659]
 [-0.00833768 -0.00833671 -0.00833571]
 [-0.00833038 -0.00832954 -0.00832862]
 [-0.00832359 -0.00832278 -0.00832203]]

results for cash allocations: 0.5
[[-0.00756311 -0.00755913 -0.007556  ]
 [-0.0075257  -0.0075232  -0.00752096]
 [-0.00751875 -0.00751853 -0.00751833]
 [-0.00751794 -0.00751674 -0.00751649]
 [-0.00751738 -0.00751594 -0.00751529]]

results for cash allocations: 0.8
[[-0.00709224 -0.00710036 -0.00710889]
 [-0.0070807  -0.00708897 -0.00709716]
 [-0.00708182 -0.00708997 -0.00709804]
 [-0.00708222 -0.00709033 -0.00709835]
 [-0.00708241 -0.00709049 -0.00709849]]

showing statistic: cvar
results for cash allocations: 0.2
[[-0.01425889 -0.01425648 -0.01425313]
 [-0.0141604  -0.01415683 -0.01415282]
 [-0.0141296  -0.01412641 -0.01412305]
 [-0.01411458 -0.

In [56]:
from plotly.subplots import make_subplots

stats_labels = ["var_alpha", "cvar", "sharpe", "sortino", "mean_return", "md",
                "stresses", "costs", "cashallocs"]

STRATEGY = total_stats_momentum

for stat_idx, stat in enumerate(stats_labels):
    img = make_subplots(rows=2, cols=2)
    zmin = np.min(STRATEGY[stat_idx])
    zmax = np.max(STRATEGY[stat_idx])
    img.add_trace(go.Heatmap(x=gammas, y=stress_corr_weights, z=STRATEGY[stat_idx, :, :, 0], zmin=zmin, zmax=zmax), row=1, col=1)
    img.add_trace(go.Heatmap(x=gammas, y=stress_corr_weights, z=STRATEGY[stat_idx, :, :, 1], zmin=zmin, zmax=zmax, showscale=False), row=1, col=2)
    img.add_trace(go.Heatmap(x=gammas, y=stress_corr_weights, z=STRATEGY[stat_idx, :, :, 2], zmin=zmin, zmax=zmax, showscale=False), row=2, col=1)
    img.add_trace(go.Heatmap(x=gammas, y=stress_corr_weights, z=STRATEGY[stat_idx, :, :, 3], zmin=zmin, zmax=zmax, showscale=False), row=2, col=2)
    img.update_layout(title=f"{stat} for MOMENTUM - X=gammas, Y=stressCorr")
    img.show()


In [57]:
stats_labels = ["var_alpha", "cvar", "sharpe", "sortino", "mean_return", "md",
                "stresses", "costs", "cashallocs"]

STRATEGY = total_stats_momentum_cvar

print('mparams used:')
for mp_val in m_params:
    print(mp_val)
for stat_idx, stat in enumerate(stats_labels):
    img = make_subplots(rows=2, cols=2)
    zmin = np.min(STRATEGY[stat_idx])
    zmax = np.max(STRATEGY[stat_idx])
    img.add_trace(go.Heatmap(x=lambdas, y=gammas, z=STRATEGY[stat_idx, :, :, 0], zmin=zmin, zmax=zmax), row=1, col=1)
    img.add_trace(go.Heatmap(x=lambdas, y=gammas, z=STRATEGY[stat_idx, :, :, 1], zmin=zmin, zmax=zmax, showscale=False), row=1, col=2)
    img.add_trace(go.Heatmap(x=lambdas, y=gammas, z=STRATEGY[stat_idx, :, :, 2], zmin=zmin, zmax=zmax, showscale=False), row=2, col=1)
    img.add_trace(go.Heatmap(x=lambdas, y=gammas, z=STRATEGY[stat_idx, :, :, 3], zmin=zmin, zmax=zmax, showscale=False), row=2, col=2)
    img.update_layout(title=f"{stat} for MOMENTUM WITH CVAR - X=lambdas, Y=gammas")
    img.show()


mparams used:
[0.33 0.33 0.33]
[0.5  0.25 0.25]
[0.25 0.5  0.25]
[0.25 0.25 0.5 ]


In [58]:

stats_labels = ["var_alpha", "cvar", "sharpe", "sortino", "mean_return", "md",
                "stresses", "costs", "cashallocs"]

STRATEGY = total_stats_er

print("Any NaNs in STRATEGY?", np.isnan(STRATEGY).any())
print("Min raw:", np.min(STRATEGY))

print('cash alloc params used:')
for cp in cash_alloc_params:
    print(cp)

for stat_idx, stat in enumerate(stats_labels):
    img = make_subplots(rows=2, cols=2)
    zmin = np.min(STRATEGY[stat_idx])
    zmax = np.max(STRATEGY[stat_idx])
    img.add_trace(go.Heatmap(x=gammas, y=stress_corr_weights, z=STRATEGY[stat_idx, :, :, 0], zmin=zmin, zmax=zmax), row=1, col=1)
    img.add_trace(go.Heatmap(x=gammas, y=stress_corr_weights, z=STRATEGY[stat_idx, :, :, 1], zmin=zmin, zmax=zmax, showscale=False), row=1, col=2)
    img.add_trace(go.Heatmap(x=gammas, y=stress_corr_weights, z=STRATEGY[stat_idx, :, :, 2], zmin=zmin, zmax=zmax, showscale=False), row=2, col=1)
    img.update_layout(title=f"{stat} for EXPECTED RETURNS - X=gammas, Y=stressCorrs")
    img.show()


Any NaNs in STRATEGY? False
Min raw: -0.1292239457097282
cash alloc params used:
0.2
0.5
0.8


In [59]:
stats_labels = ["var_alpha", "cvar", "sharpe", "sortino", "mean_return", "md",
                "stresses", "costs", "cashallocs"]

STRATEGY = total_stats_er_cvar

print('stress_corrparams used:')
for cp in stress_corr_weights:
    print(cp)

for stat_idx, stat in enumerate(stats_labels):
    img = make_subplots(rows=2, cols=2)
    zmin = np.min(STRATEGY[stat_idx])
    zmax = np.max(STRATEGY[stat_idx])
    img.add_trace(go.Heatmap(x=lambdas, y=gammas, z=STRATEGY[stat_idx, :, :, 0], zmin=zmin, zmax=zmax), row=1, col=1)
    img.add_trace(go.Heatmap(x=lambdas, y=gammas, z=STRATEGY[stat_idx, :, :, 1], zmin=zmin, zmax=zmax, showscale=False), row=1, col=2)
    img.add_trace(go.Heatmap(x=lambdas, y=gammas, z=STRATEGY[stat_idx, :, :, 2], zmin=zmin, zmax=zmax, showscale=False), row=2, col=1)
    img.update_layout(title=f"{stat} for EXPECTED RETURNS WITH CVAR - X=lambdas, Y=gammas")
    img.show()


stress_corrparams used:
0.1
0.2
0.3


In [62]:
stats_labels = ["var_alpha", "cvar", "sharpe", "sortino", "mean_return", "md",
                "stresses", "costs", "cashallocs"]

STRATEGY = total_stats_sharpe

print('cash params used:')
for cp in cash_alloc_params:
    print(cp)

for stat_idx, stat in enumerate(stats_labels):
    img = make_subplots(rows=2, cols=2)
    zmin = np.min(STRATEGY[stat_idx])
    zmax = np.max(STRATEGY[stat_idx])
    img.add_trace(go.Heatmap(x=gammas, y=stress_corr_weights, z=STRATEGY[stat_idx, :, :, 0], zmin=zmin, zmax=zmax), row=1, col=1)
    img.add_trace(go.Heatmap(x=gammas, y=stress_corr_weights, z=STRATEGY[stat_idx, :, :, 1], zmin=zmin, zmax=zmax, showscale=False), row=1, col=2)
    img.add_trace(go.Heatmap(x=gammas, y=stress_corr_weights, z=STRATEGY[stat_idx, :, :, 2], zmin=zmin, zmax=zmax, showscale=False), row=2, col=1)
    img.update_layout(title=f"{stat} for SHARPE OPTIMALIZATION - X=gamma, Y=stress")
    img.show()


cash params used:
0.2
0.5
0.8


In [63]:
stats_labels = ["var_alpha", "cvar", "sharpe", "sortino", "mean_return", "md",
                "stresses", "costs", "cashallocs"]

STRATEGY = total_stats_sharpe_cvar

print('stress values used:')
for cp in stress_corr_weights:
    print(cp)

for stat_idx, stat in enumerate(stats_labels):
    img = make_subplots(rows=2, cols=2)
    zmin = np.min(STRATEGY[stat_idx])
    zmax = np.max(STRATEGY[stat_idx])
    img.add_trace(go.Heatmap(x=lambdas, y=gammas, z=STRATEGY[stat_idx, :, :, 0], zmin=zmin, zmax=zmax), row=1, col=1)
    img.add_trace(go.Heatmap(x=lambdas, y=gammas, z=STRATEGY[stat_idx, :, :, 1], zmin=zmin, zmax=zmax, showscale=False), row=1, col=2)
    img.add_trace(go.Heatmap(x=lambdas, y=gammas, z=STRATEGY[stat_idx, :, :, 2], zmin=zmin, zmax=zmax, showscale=False), row=2, col=1)
    img.update_layout(title=f"{stat} for SHARPE OPTIMALIZATION WITH CVAR - X=lambda, Y=gamma")
    img.show()


stress values used:
0.1
0.2
0.3


In [ ]:
"""
CONCLUSION

FOR MOMENTUM BASED:


FOR SHARPE BASED:



"""